# 1. Library


In [ ]:
import os
import gc
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, optimizers
from tensorflow.keras.utils import image_dataset_from_directory
from tensorflow.keras import mixed_precision
from tensorflow.keras import optimizers


# 2. Basic configuration


In [ ]:
import os
import gdown

# 1. Setup
file_id = '1AlEKzyLZzfIuUza_OlS2ulGLWBFUahyX' # <--- PASTE THE ID HERE
url = f'https://drive.google.com/uc?id={file_id}'
output = 'Fruit-262.zip'

# 2. Download directly to Colab
print(f"⬇️ Downloading file with ID: {file_id}...")
gdown.download(url, output, quiet=False)

# 3. Unzip
if os.path.exists(output):
    print(f"✅ Download successful: {output}")
    print("🚀 Unzipping...")

    # Unzip to a local folder
    !unzip -q "{output}" -d "/content/fruits_dataset"
    print("✅ Done! Dataset ready at /content/fruits_dataset")
else:
    print("❌ Error: Download failed. Check the File ID.")

⬇️ Downloading file with ID: 1AlEKzyLZzfIuUza_OlS2ulGLWBFUahyX...


Downloading...
From (original): https://drive.google.com/uc?id=1AlEKzyLZzfIuUza_OlS2ulGLWBFUahyX
From (redirected): https://drive.google.com/uc?id=1AlEKzyLZzfIuUza_OlS2ulGLWBFUahyX&confirm=t&uuid=ac9c1c83-329e-48ed-89f4-dfda9d951b0c
To: /content/Fruit-262.zip
100%|██████████| 2.92G/2.92G [00:39<00:00, 74.9MB/s]


✅ Download successful: Fruit-262.zip
🚀 Unzipping...
✅ Done! Dataset ready at /content/fruits_dataset


# 3. Data preprocessing


In [ ]:
# 2. CONFIG
# Mixed Precision is CRITICAL for speed on T4 GPUs
mixed_precision.set_global_policy('mixed_float16')

DATA_ROOT = "/content/fruits_dataset" # Explicit path is safer
IMG_SIZE = (160, 160)
BATCH_SIZE = 64  # Increased to 64. MobileNet is small, so this reduces overhead.

# 3. FAST DATA LOADING
train_ds = image_dataset_from_directory(
    DATA_ROOT,
    validation_split=0.2,
    subset="training",
    seed=123,
    labels="inferred",
    label_mode="int",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_ds = image_dataset_from_directory(
    DATA_ROOT,
    validation_split=0.2,
    subset="validation",
    seed=123,
    labels="inferred",
    label_mode="int",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

class_names = train_ds.class_names
NUM_CLASSES = len(class_names)

# 4. OPTIMIZED PIPELINE
AUTOTUNE = tf.data.AUTOTUNE

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
])

def prepare(ds, training=False):
    # Standard MobileNetV2 preprocessing
    ds = ds.map(lambda x, y: (tf.keras.applications.mobilenet_v2.preprocess_input(x), y),
                num_parallel_calls=AUTOTUNE) # Enable Parallel CPU processing

    if training:
        ds = ds.map(lambda x, y: (data_augmentation(x, training=True), y),
                    num_parallel_calls=AUTOTUNE)

    # ✅ FIX: Use AUTOTUNE here. This allows CPU and GPU to work at the same time.
    return ds.prefetch(buffer_size=AUTOTUNE)

train_ds_proc = prepare(train_ds, training=True)
val_ds_proc = prepare(val_ds, training=False)

# Clear old models from RAM
tf.keras.backend.clear_session()
gc.collect()


Found 225639 files belonging to 262 classes.
Using 180512 files for training.
Found 225639 files belonging to 262 classes.
Using 45127 files for validation.


0

# 4. MobileNetV2

In [ ]:
# 5. BUILD MODEL
print(f"--- Building MobileNetV2 for {NUM_CLASSES} classes ---")
base_model = tf.keras.applications.MobileNetV2(
    input_shape=IMG_SIZE + (3,),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False

inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
x = layers.Dense(NUM_CLASSES)(x)
outputs = layers.Activation('softmax', dtype='float32')(x)

model = models.Model(inputs, outputs)

model.compile(
    optimizer=optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# 6. TRAIN
print("--- Starting Training ---")
history = model.fit(
    train_ds_proc,
    validation_data=val_ds_proc,
    epochs=4,
    callbacks=[callbacks.EarlyStopping(patience=3, restore_best_weights=True)]
)

# 7. EXPORT
print("--- Exporting to TFLite ---")
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

with open('fruit_model.tflite', 'wb') as f:
    f.write(tflite_model)
with open('labels.txt', 'w') as f:
    f.write('\n'.join(class_names))

try:
    from google.colab import files
    files.download('fruit_model.tflite')
    files.download('labels.txt')
except:
    pass

TensorFlow version: 2.19.0
✅ GPU Active: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Found 225639 files belonging to 262 classes.
Using 180512 files for training.
Found 225639 files belonging to 262 classes.
Using 45127 files for validation.
--- Building MobileNetV2 for 262 classes ---
--- Starting Training ---
Epoch 1/4
2821/2821 ━━━━━━━━━━━━━━━━━━━━ 716s 251ms/step - accuracy: 0.3596 - loss: 2.9006 - val_accuracy: 0.6630 - val_loss: 1.3746
Epoch 2/4
2821/2821 ━━━━━━━━━━━━━━━━━━━━ 790s 270ms/step - accuracy: 0.5446 - loss: 1.8518 - val_accuracy: 0.6990 - val_loss: 1.2146
Epoch 3/4
2821/2821 ━━━━━━━━━━━━━━━━━━━━ 696s 247ms/step - accuracy: 0.5696 - loss: 1.7423 - val_accuracy: 0.7086 - val_loss: 1.1689
Epoch 4/4
2821/2821 ━━━━━━━━━━━━━━━━━━━━ 743s 263ms/step - accuracy: 0.5782 - loss: 1.6931 - val_accuracy: 0.7152 - val_loss: 1.1425
--- Exporting to TFLite ---
Saved artifact at '/tmp/tmpaqr19xzi'. The following endpoints are available:

* Endpoint 'serve'
  args_

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ==========================================
# STAGE 2: FINE-TUNING (The "Unfreeze" Trick)
# ==========================================
print("\n--- 🔓 Unfreezing top layers for Fine-Tuning ---")

# 1. Unfreeze the base model
base_model.trainable = True

# 2. Refreeze the bottom layers (Keep the core generic features locked)
# MobileNetV2 has 154 layers. Let's fine-tune the top 50.
fine_tune_at = 100
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

# 3. CRITICAL: Recompile with a LOW Learning Rate
# If you use the default 0.001, you will destroy the brain! Use 0.00001 (1e-5).
model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-5), # <--- Very slow, careful updates
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# 4. Continue Training
print("--- Starting Fine-Tuning ---")
history_fine = model.fit(
    train_ds_proc,
    validation_data=val_ds_proc,
    initial_epoch=history.epoch[-1], # Start where we left off
    epochs=10, # Add 5-10 more epochs
    callbacks=[callbacks.EarlyStopping(patience=3, restore_best_weights=True)]
)


--- 🔓 Unfreezing top layers for Fine-Tuning ---
--- Starting Fine-Tuning ---
Epoch 4/10
2821/2821 ━━━━━━━━━━━━━━━━━━━━ 750s 257ms/step - accuracy: 0.4305 - loss: 2.6789 - val_accuracy: 0.6979 - val_loss: 1.1975
Epoch 5/10
2821/2821 ━━━━━━━━━━━━━━━━━━━━ 705s 249ms/step - accuracy: 0.5790 - loss: 1.6435 - val_accuracy: 0.7324 - val_loss: 1.0539
Epoch 6/10
2821/2821 ━━━━━━━━━━━━━━━━━━━━ 697s 247ms/step - accuracy: 0.6222 - loss: 1.4570 - val_accuracy: 0.7555 - val_loss: 0.9677
Epoch 7/10
2821/2821 ━━━━━━━━━━━━━━━━━━━━ 697s 247ms/step - accuracy: 0.6478 - loss: 1.3514 - val_accuracy: 0.7697 - val_loss: 0.9137
Epoch 8/10
2821/2821 ━━━━━━━━━━━━━━━━━━━━ 740s 246ms/step - accuracy: 0.6663 - loss: 1.2726 - val_accuracy: 0.7825 - val_loss: 0.8589
Epoch 9/10
2821/2821 ━━━━━━━━━━━━━━━━━━━━ 695s 246ms/step - accuracy: 0.6801 - loss: 1.2114 - val_accuracy: 0.7934 - val_loss: 0.8160
Epoch 10/10
2821/2821 ━━━━━━━━━━━━━━━━━━━━ 778s 276ms/step - accuracy: 0.6922 - loss: 1.1634 - val_accuracy: 0.7964 - 

In [ ]:
print("--- 📥 Exporting & Downloading ---")
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

# Save to disk
with open('fruit_model_v2.tflite', 'wb') as f:
    f.write(tflite_model)

# Download to your computer
from google.colab import files
files.download('fruit_model_v2.tflite')
files.download('labels.txt')

In [ ]:
import tensorflow as tf
import numpy as np
import os
from tensorflow.keras.models import load_model
from tensorflow.keras.utils import image_dataset_from_directory

# ==========================================
# 1. SETUP
# ==========================================
# ✅ Point to the 262-class folder
TEST_DIR = "/content/fruits_dataset"
MODEL_PATH = "fruit_mobilenet_v2_epoch30.h5"
IMG_SIZE = (160, 160)
BATCH_SIZE = 32

# ==========================================
# 2. LOAD DATA
# ==========================================
print("--- 🚀 Loading Test Data ---")
test_ds = image_dataset_from_directory(
    TEST_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True, # Shuffle for a good random sample
    seed=42
)

# ==========================================
# 3. CRITICAL: APPLY PREPROCESSING
# ==========================================
# We need to transform 0-255 pixels into the range the model expects.
# MobileNetV2 expects [-1, 1] or [0, 1]. Let's use the official helper.

preprocess_input = tf.keras.applications.mobilenet_v2.preprocess_input

print("--- ⚙️ Applying MobileNetV2 Preprocessing ---")
def preprocess(image, label):
    # This converts 0-255 to -1 to 1 (Standard for MobileNetV2)
    return preprocess_input(image), label

test_ds_proc = test_ds.map(preprocess)

# ==========================================
# 4. EVALUATE
# ==========================================
print(f"--- 🧠 Loading {MODEL_PATH} ---")
model = load_model(MODEL_PATH)

print("--- ⚡ Running Evaluation ---")
loss, accuracy = model.evaluate(test_ds_proc)

print("\n" + "="*40)
print(f"📊 Dataset: 262 Classes")
print(f"🏆 FINAL ACCURACY: {accuracy * 100:.2f}%")
print(f"📉 Loss: {loss:.4f}")
print("="*40)

--- 🚀 Loading Test Data ---
Found 225639 files belonging to 262 classes.
--- ⚙️ Applying MobileNetV2 Preprocessing ---
--- 🧠 Loading fruit_mobilenet_v2_epoch30.h5 ---


--- ⚡ Running Evaluation ---
7052/7052 ━━━━━━━━━━━━━━━━━━━━ 240s 32ms/step - accuracy: 0.8340 - loss: 0.6151

📊 Dataset: 262 Classes
🏆 FINAL ACCURACY: 83.48%
📉 Loss: 0.6118
